# Murmur Mamba3 SISO — 40M smoke on Kaggle T4

A from-scratch control run for the recurrent Mamba3 SISO Core. It does not modify the preserved GQA or experimental MIMO paths. Enable GPU and Internet; attach `almirneto/corpus-c4` through **Add Data**.


In [ ]:
# 1. Project environment — clone the exact SISO experiment branch
from pathlib import Path
import subprocess, sys
repo_dir = Path('/kaggle/working/science')
if not (repo_dir / 'pyproject.toml').exists():
    subprocess.check_call(['git', 'clone', '--branch', 'codex/mamba3-siso-smoke', '--single-branch', 'https://github.com/orkrs/murmur-science.git', str(repo_dir)])
%cd /kaggle/working/science
%pip install -q --no-deps -e .
sys.path[:0] = ['/kaggle/working/science/src', '/kaggle/working/science']
Path('artifacts').mkdir(exist_ok=True)
print('Environment ready')

In [ ]:
# 2. GPU and official Mamba3 SISO installation
import os, torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable Kaggle GPU: Settings -> Accelerator -> GPU, then restart.')
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))
!nvidia-smi
# SISO uses Triton, not TileLang. Remove TileLang so Mamba3's optional MIMO import
# raises ImportError (which upstream handles) instead of its broken TVM-FFI AttributeError.
%pip uninstall -q -y tilelang apache-tvm-ffi
%pip install -q --upgrade einops
os.environ['MAMBA_FORCE_BUILD'] = 'TRUE'
# --no-deps preserves Kaggle's CUDA-enabled PyTorch instead of reinstalling a CPU wheel.
%pip install -q --no-cache-dir --no-deps --force-reinstall --no-build-isolation git+https://github.com/state-spaces/mamba.git@main
!python scripts/hardware_probe.py --output artifacts/hardware_mamba3_siso.json

In [ ]:
# 3. Mandatory SISO gate: real forward + backward on this T4
import json
from mamba_ssm.modules.mamba3 import Mamba3
try:
    gate = Mamba3(d_model=128, d_state=64, headdim=64, is_mimo=False, chunk_size=64).cuda().half().train()
    x = torch.randn(1, 128, 128, device='cuda', dtype=torch.float16, requires_grad=True)
    y = gate(x)
    y.float().square().mean().backward()
    if not torch.isfinite(y).all() or not torch.isfinite(x.grad).all():
        raise RuntimeError('SISO forward/backward returned non-finite tensors')
    gate_result = {'passed': True, 'gpu': torch.cuda.get_device_name(0), 'output_shape': list(y.shape)}
except Exception as exc:
    gate_result = {'passed': False, 'error': repr(exc)}
Path('artifacts/mamba3_siso_gate.json').write_text(json.dumps(gate_result, indent=2))
print(gate_result)
if not gate_result['passed']:
    raise RuntimeError('Mamba3 SISO gate failed. Stop and send artifacts/mamba3_siso_gate.json.')

In [ ]:
# 4. Verify the control configuration
!python scripts/param_count.py --config configs/smoke_mamba3_siso.toml
from murmur.config import load_run_config
config = load_run_config(Path('configs/smoke_mamba3_siso.toml'))
assert config.model.mixer == 'mamba3'
print('Verified: recurrent Mamba3 SISO Core; Prelude/Coda remain GQA.')

In [ ]:
# 5. Build a deterministic 20M-token C4 smoke corpus
import hashlib, json, re
import pandas as pd
raw_candidates = [p for p in Path('/kaggle/input').iterdir() if p.is_dir() and ('c4' in p.name.lower() or 'corpus' in p.name.lower())]
if not raw_candidates:
    raise FileNotFoundError('Attach almirneto/corpus-c4 through Add Data, then re-run this cell.')
raw_dir = raw_candidates[0]
out = Path('artifacts/siso_smoke_corpus'); out.mkdir(parents=True, exist_ok=True)
budget, accepted, seen, train, val = 20_000_000, 0, set(), [], []
def add(text):
    global accepted
    text = re.sub(r'\s+', ' ', str(text)).strip()
    if len(text) < 80: return
    digest = hashlib.sha256(text.encode()).hexdigest(); estimate = max(1, len(text.encode()) // 4)
    if digest in seen or accepted + estimate > budget: return
    seen.add(digest); accepted += estimate
    row = {'text': text, 'source': 'almirneto/corpus-c4', 'text_sha256': digest}
    (val if int(digest[:8], 16) % 100 < 2 else train).append(row)
for path in sorted(raw_dir.rglob('*')):
    if accepted >= budget: break
    if path.suffix.lower() == '.csv':
        for chunk in pd.read_csv(path, chunksize=10_000):
            column = next((c for c in ('text','content','body') if c in chunk.columns), None)
            if column:
                for text in chunk[column].dropna():
                    add(text)
                    if accepted >= budget: break
    elif path.suffix.lower() == '.parquet':
        frame = pd.read_parquet(path)
        column = next((c for c in ('text','content','body') if c in frame.columns), None)
        if column:
            for text in frame[column].dropna():
                add(text)
                if accepted >= budget: break
for name, rows in [('train.jsonl', train), ('val.jsonl', val)]:
    with (out / name).open('w', encoding='utf-8') as f:
        for row in rows: f.write(json.dumps(row, ensure_ascii=False) + '\n')
with (out / 'corpus.txt').open('w', encoding='utf-8') as f:
    for row in train + val: f.write(row['text'] + '\n\n')
if not train or not val: raise RuntimeError('Corpus conversion produced an empty split')
print({'train_docs': len(train), 'val_docs': len(val), 'estimated_tokens': accepted})

In [ ]:
# 6. Fresh tokenizer and packed data
!python scripts/train_tokenizer.py --corpus artifacts/siso_smoke_corpus/corpus.txt --output artifacts/siso_smoke_tokenizer.model --vocab-size 32000
!python scripts/prepare_data.py --config configs/smoke_mamba3_siso.toml --tokenizer artifacts/siso_smoke_tokenizer.model --train-input artifacts/siso_smoke_corpus/train.jsonl --val-input artifacts/siso_smoke_corpus/val.jsonl --output artifacts/siso_smoke_data
assert list(Path('artifacts/siso_smoke_data').glob('train_*.bin'))
assert list(Path('artifacts/siso_smoke_data').glob('val_*.bin'))
print('Packed smoke data ready')

In [ ]:
# 7. Train from random weights — 10M tokens
template = Path('configs/smoke_mamba3_siso.toml').read_text()
session = template.replace('artifacts/data/train.bin', 'artifacts/siso_smoke_data/train.bin').replace('artifacts/data/val.bin', 'artifacts/siso_smoke_data/val.bin')
Path('configs/smoke_mamba3_siso_session.toml').write_text(session)
run_dir = Path('artifacts/runs/mamba3_siso_40m_smoke')
subprocess.run(['python', 'scripts/train.py', '--config', 'configs/smoke_mamba3_siso_session.toml', '--run-dir', str(run_dir), '--device', 'cuda'], check=True)
assert (run_dir / 'checkpoints' / 'last' / 'COMPLETED').exists()
print('Mamba3 SISO smoke checkpoint ready')

In [ ]:
# 8. Evaluate. Generation is deferred until recurrent Mamba cache parity is implemented.
!python scripts/evaluate.py --config configs/smoke_mamba3_siso_session.toml --checkpoint artifacts/runs/mamba3_siso_40m_smoke/checkpoints/last --output artifacts/eval_mamba3_siso_40m.json --device cuda
print(Path('artifacts/eval_mamba3_siso_40m.json').read_text())
print('Send the gate JSON, training log and evaluation JSON for comparison with GQA.')